# Phase 1 - Dataset Discovery

A notebook that discovers candidate datasets for the financial-discussion surge-prediction project by querying the **Kaggle** and **HuggingFace** dataset APIs, then ranks them into a **draft candidate list**.

It also includes a short **reference note** on collecting fresh data via the X/Twitter and Reddit APIs, with links to the official pricing/terms pages.

## What this notebook is (and is not)

- It **is** a coarse, title-keyword-based funnel. The Kaggle/HF search APIs rarely expose column schemas, so completeness flags (`has_engagement_metrics`, `has_sentiment_fields`, `is_complete`) are inferred from **titles/tags**, not real columns. Expect false positives and misses.
- It is **not** a precise validator. Real per-dataset validation happens in **Phase 2** (high-level eval) and **Phase 3** (deep assessment).

**Output (CSV):** `src/eda/output/candidates.csv`, a draft candidate list of available datasets.

This notebook does not import any project `.py` modules; all logic is inline.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SETUP dependencies (uncomment if needed)
# ══════════════════════════════════════════════════════════════════════════════
# !pip install kaggle huggingface_hub pandas
#
# Credentials (only needed for LIVE discovery; the notebook degrades gracefully):
#   Kaggle:      place kaggle.json at ~/.kaggle/kaggle.json  (or set KAGGLE_USERNAME / KAGGLE_KEY)
#   HuggingFace: usually works anonymously for public dataset search

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import json
import logging
import math
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("discovery")

print("Setup complete.")

Setup complete.


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# Resolve project root robustly whether run from src/eda or project root.
current_dir = Path.cwd()
if (current_dir / "src" / "eda").exists():
    PROJECT_ROOT = current_dir
elif current_dir.name == "eda" and (current_dir.parent.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent.parent
else:
    PROJECT_ROOT = current_dir

OUTPUT_DIR = PROJECT_ROOT / "src" / "eda" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Search terms used against both Kaggle and HuggingFace.
SEARCH_TERMS = [
    "twitter finance",
    "reddit finance"   
]

# Filter / ranking thresholds.
MIN_DOWNLOAD_COUNT = 1000     # keep datasets with >= this many downloads (0 = no filter)
MAX_FRESHNESS_DAYS = -1       # keep datasets updated within N days (<=0 = no limit)
REQUIRE_COMPLETE = False      # if True, keep only datasets flagged engagement AND sentiment
TOP_K = 10                    # keep top-K PER PLATFORM after ranking (0 = keep all)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")
print(f"Search terms: {SEARCH_TERMS}")

PROJECT_ROOT: d:\git\uol-bsc-cm3070-final-project
OUTPUT_DIR:   d:\git\uol-bsc-cm3070-final-project\src\eda\output
Search terms: ['twitter finance', 'reddit finance']


---
## 1. Data contract + keyword heuristics

`DatasetMetadata` is copied inline (no shared module). Completeness is decided in **one place** (`_evaluate_completeness`) to avoid the flag-flip bug where different call sites computed it differently.

In [14]:
@dataclass
class DatasetMetadata:
    """Metadata for a discovered dataset (inline copy; non-frozen so flags can update)."""
    name: str
    source_platform: str
    record_count: int = 0
    download_count: int = 0
    date_range: tuple[str, str] = ("unknown", "unknown")
    columns: list[str] = field(default_factory=list)
    freshness_days: int = -1
    has_engagement_metrics: bool = False
    has_sentiment_fields: bool = False
    is_complete: bool = False


ENGAGEMENT_KEYWORDS: set[str] = {
    "likes", "retweets", "comments", "upvotes", "shares", "favorites",
    "score", "num_comments", "comment_count", "like_count", "retweet_count",
}
SENTIMENT_KEYWORDS: set[str] = {
    "sentiment", "polarity", "bullish", "bearish", "positive", "negative",
    "sentiment_score",
}
ENGAGEMENT_TITLE_HINTS = ("engagement", "likes", "retweets", "comments", "upvotes")
SENTIMENT_TITLE_HINTS = ("sentiment", "polarity", "bullish", "bearish")


def _check_engagement(columns: list[str], title: str = "") -> bool:
    if columns:
        return bool({c.lower() for c in columns} & ENGAGEMENT_KEYWORDS)
    return any(k in title.lower() for k in ENGAGEMENT_TITLE_HINTS)


def _check_sentiment(columns: list[str], title: str = "") -> bool:
    if columns:
        return bool({c.lower() for c in columns} & SENTIMENT_KEYWORDS)
    return any(k in title.lower() for k in SENTIMENT_TITLE_HINTS)


def _evaluate_completeness(md: DatasetMetadata, title_for_fallback: str) -> None:
    """Single source of truth for completeness flags. Mutates md in place.

    Uses real columns when available; otherwise falls back to the SAME title text
    for both engagement and sentiment checks (prevents scanner-vs-postprocess drift).
    """
    title = title_for_fallback if not md.columns else ""
    md.has_engagement_metrics = _check_engagement(md.columns, title)
    md.has_sentiment_fields = _check_sentiment(md.columns, title)
    md.is_complete = md.has_engagement_metrics and md.has_sentiment_fields


def _freshness_days(dt: datetime) -> int:
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return max(0, (datetime.now(timezone.utc) - dt).days)


def _parse_dt(value: Any) -> Optional[datetime]:
    if value is None:
        return None
    if isinstance(value, datetime):
        return value
    if isinstance(value, str):
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except (ValueError, TypeError):
            return None
    return None


print("Data contract + heuristics defined.")

Data contract + heuristics defined.


> **Note:** Kaggle/HF search endpoints seldom return column schemas. When columns are unavailable, `is_complete` is inferred from title/tag keywords only. This is a *coarse funnel*; treat the output as a **draft** to be validated in Phase 2/3, not as ground truth.

---
## 2. Kaggle scan (inline)

Returns `[]` gracefully if the `kaggle` package, credentials, or network are unavailable.

In [ ]:
def scan_kaggle(search_terms: list[str]) -> list[DatasetMetadata]:
    """Search Kaggle for datasets matching search_terms. Empty list on any failure."""
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
    except Exception as e:
        logger.warning("Kaggle API unavailable (%s). Skipping Kaggle scan.", e)
        return []

    seen: set[str] = set()
    out: list[DatasetMetadata] = []

    for term in search_terms:
        try:
            results = api.dataset_list(search=term)
        except Exception as e:
            logger.warning("Kaggle search error for '%s': %s", term, e)
            continue

        for ds in results or []:
            name = getattr(ds, "ref", None) or getattr(ds, "title", str(ds))
            if name in seen:
                continue
            seen.add(name)

            # Attribute names differ across kaggle SDK versions: newer kagglesdk uses
            # snake_case (download_count, last_updated); older kaggle used camelCase.
            def _attr(obj, *names, default=None):
                for n in names:
                    v = getattr(obj, n, None)
                    if v is not None:
                        return v
                return default

            download_count = _attr(ds, "download_count", "downloadCount", default=0) or 0
            last_updated = _parse_dt(_attr(ds, "last_updated", "lastUpdated"))
            if last_updated is not None:
                freshness = _freshness_days(last_updated)
                end_date = last_updated.strftime("%Y-%m-%d")
            else:
                freshness, end_date = -1, "unknown"

            # NOTE: api.dataset_list_files() is intentionally NOT called per dataset.
            # It fires an extra API round-trip for every result (hundreds across all
            # search terms), making the scan slow and prone to timeouts/partial results,
            # while rarely returning usable column info. Completeness falls back to the
            # title keywords; real column/schema validation happens in Phase 2/3.
            columns: list[str] = []

            title = getattr(ds, "title", name) or name
            md = DatasetMetadata(
                name=name,
                source_platform="kaggle",
                record_count=0,  # not reliably exposed by search API
                download_count=download_count,
                date_range=("unknown", end_date),
                columns=columns,
                freshness_days=freshness,
            )
            _evaluate_completeness(md, title)
            out.append(md)

    logger.info("Kaggle: %d unique candidate(s).", len(out))
    return out


print("scan_kaggle defined.")

scan_kaggle defined.


---
## 3. HuggingFace scan (inline)

**Bug fix applied:** the original code assigned `dataset_size` (a byte count) to `record_count`. That is dropped here; `record_count` stays `0` unless an actual row count is found in split metadata (`num_examples` / `num_rows`).

In [16]:
def scan_huggingface(search_terms: list[str]) -> list[DatasetMetadata]:
    """Search HuggingFace for datasets matching search_terms. Empty list on any failure."""
    try:
        from huggingface_hub import HfApi, list_datasets
    except ImportError as e:
        logger.warning("huggingface_hub unavailable (%s). Skipping HF scan.", e)
        return []

    seen: set[str] = set()
    out: list[DatasetMetadata] = []
    api = HfApi()

    for term in search_terms:
        try:
            results = list(list_datasets(search=term, limit=50))
        except Exception as e:
            logger.warning("HuggingFace search error for '%s': %s", term, e)
            continue

        for ds in results or []:
            name = getattr(ds, "id", None) or str(ds)
            if name in seen:
                continue
            seen.add(name)

            download_count = getattr(ds, "downloads", 0) or 0
            last_mod = _parse_dt(
                getattr(ds, "lastModified", None) or getattr(ds, "last_modified", None)
            )
            if last_mod is not None:
                freshness = _freshness_days(last_mod)
                end_date = last_mod.strftime("%Y-%m-%d")
            else:
                freshness, end_date = -1, "unknown"

            columns: list[str] = []
            record_count = 0
            try:
                info = api.dataset_info(name)
                card = getattr(info, "card_data", None)
                if card:
                    features = getattr(card, "features", None)
                    if isinstance(features, dict):
                        columns = list(features.keys())
                    # Row count ONLY from explicit split example counts (NOT dataset_size, which is bytes).
                    configs = getattr(card, "configs", None) or getattr(card, "dataset_info", None)
                    if isinstance(configs, list):
                        for cfg in configs:
                            if isinstance(cfg, dict):
                                for split in cfg.get("splits", []) or []:
                                    if isinstance(split, dict):
                                        record_count += (
                                            split.get("num_examples", 0)
                                            or split.get("num_rows", 0)
                                        )
            except Exception:
                logger.debug("No detailed metadata for HF dataset %s", name)

            tags = getattr(ds, "tags", []) or []
            title = f"{name} {' '.join(tags)}" if tags else name
            md = DatasetMetadata(
                name=name,
                source_platform="huggingface",
                record_count=record_count,
                download_count=download_count,
                date_range=("unknown", end_date),
                columns=columns,
                freshness_days=freshness,
            )
            _evaluate_completeness(md, title)
            out.append(md)

    logger.info("HuggingFace: %d unique candidate(s).", len(out))
    return out


print("scan_huggingface defined.")

scan_huggingface defined.


---
## 4. Filter + rank (inline)

In [17]:
def filter_datasets(
    datasets: list[DatasetMetadata],
    *,
    require_complete: bool = False,
    min_download_count: int = 0,
    max_freshness_days: int = -1,
    top_k: int = 0,
) -> list[DatasetMetadata]:
    """Apply hard filters, then rank by a relevance score and keep top_k."""
    filtered = list(datasets)

    if require_complete:
        filtered = [d for d in filtered if d.is_complete]
    if min_download_count > 0:
        filtered = [d for d in filtered if d.download_count >= min_download_count]
    if max_freshness_days > 0:
        filtered = [d for d in filtered if 0 <= d.freshness_days <= max_freshness_days]

    # Neutral baselines so datasets whose metadata the search API does NOT report
    # (download_count -> 0, freshness_days -> -1) are not unfairly buried at score 0.
    NEUTRAL_POPULARITY = 5.0   # ~ a dataset with 10^0.5 downloads

    def _score(d: DatasetMetadata) -> float:
        score = 0.0
        if d.is_complete:
            score += 50.0
        elif d.has_engagement_metrics or d.has_sentiment_fields:
            score += 20.0
        # Popularity: use log-scaled downloads when known, else a neutral baseline.
        if d.download_count > 0:
            score += math.log10(d.download_count + 1) * 10.0
        else:
            score += NEUTRAL_POPULARITY
        # Freshness intentionally NOT scored: this project targets a fixed 2021-era
        # archive, so recency is not a relevance signal and old datasets must not be
        # penalised. (MAX_FRESHNESS_DAYS can still hard-filter if explicitly set.)
        if d.record_count > 0:
            score += 10.0
        return score

    filtered.sort(key=_score, reverse=True)
    if top_k > 0:
        filtered = filtered[:top_k]
    return filtered


print("filter_datasets defined.")

filter_datasets defined.


---
## 5. Run discovery

Live discovery only. If neither API is reachable (missing packages, credentials, or network), the candidate list will be empty and the notebook prints a clear notice, fix credentials/network and re-run. Dataset selection is manual downstream, so no offline seed fallback is used.

In [18]:
kaggle_hits = scan_kaggle(SEARCH_TERMS)
hf_hits = scan_huggingface(SEARCH_TERMS)
combined = kaggle_hits + hf_hits
matched_count = len(combined)  # total raw matches before TOP_K truncation

if not combined:
    print("\n" + "=" * 60)
    print("NOTICE: No datasets returned from Kaggle or HuggingFace.")
    print("Check that the kaggle / huggingface_hub packages are installed,")
    print("credentials are configured, and the network is reachable, then re-run.")
    print("=" * 60)

# Rank and keep TOP_K PER PLATFORM (e.g. 5 Kaggle + 5 HuggingFace).
def _rank_platform(hits):
    return filter_datasets(
        hits,
        require_complete=REQUIRE_COMPLETE,
        min_download_count=MIN_DOWNLOAD_COUNT,
        max_freshness_days=MAX_FRESHNESS_DAYS,
        top_k=TOP_K,
    )

ranked_kaggle = _rank_platform(kaggle_hits)
ranked_hf = _rank_platform(hf_hits)
ranked = ranked_kaggle + ranked_hf
retained_count = len(ranked)
truncated = matched_count > retained_count

print(f"\nMatched: {matched_count} (kaggle={len(kaggle_hits)}, hf={len(hf_hits)})")
print(f"Retained TOP_K={TOP_K} per platform: {retained_count} "
      f"(kaggle={len(ranked_kaggle)}, hf={len(ranked_hf)})")
if truncated:
    print(f"LIMITATION: {matched_count - retained_count} lower-ranked match(es) dropped by per-platform TOP_K={TOP_K}.")

INFO: Kaggle: 37 unique candidate(s).
INFO: HTTP Request: GET https://huggingface.co/api/datasets?search=twitter+finance&limit=50 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets?search=reddit+finance&limit=50 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/winddude/reddit_finance_43_250k "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/egupta/reddit-finance-qa-json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/aurelio-ai/reddit-finance "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/emilpartow/reddit_finance_posts_apple-tesla-microsoft "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/emilpartow/reddit_finance_posts_sp500 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/Nitish-Garikoti/reddit_finance_43_250k "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/datas


Matched: 47 (kaggle=37, hf=10)
Retained TOP_K=10 per platform: 10 (kaggle=10, hf=0)
LIMITATION: 37 lower-ranked match(es) dropped by per-platform TOP_K=10.


In [19]:
# Present the retained candidates as a table.
if ranked:
    candidates_df = pd.DataFrame([asdict(d) for d in ranked])
    display_cols = [
        "name", "source_platform", "download_count", "record_count",
        "freshness_days", "has_engagement_metrics", "has_sentiment_fields", "is_complete",
    ]
    print(f"Showing top {retained_count} of {matched_count} matched dataset(s):")
    display(candidates_df[display_cols])
else:
    candidates_df = pd.DataFrame()
    print("No candidates to display.")

Showing top 10 of 47 matched dataset(s):


,name,source_platform,download_count,record_count,freshness_days,has_engagement_metrics,has_sentiment_fields,is_complete
0,yash612/stockmarket-sentiment-dataset,kaggle,14073,0,2276,False,True,False
1,sidarcidiacono/news-sentiment-analysis-for-sto...,kaggle,1189,0,1991,False,True,False
2,kemical/kickstarter-projects,kaggle,99893,0,3125,False,False,False
3,aaron7sun/stocknews,kaggle,64527,0,2481,False,False,False
4,gpreda/reddit-wallstreetsbets-posts,kaggle,18603,0,1840,False,False,False
5,davidwallach/financial-tweets,kaggle,14356,0,2942,False,False,False
6,pratyushpuri/multilingual-mobile-app-reviews-d...,kaggle,7689,0,395,False,False,False
7,arathee2/demonetization-in-india-twitter-data,kaggle,7334,0,3417,False,False,False
8,hananxx/gamestop-historical-stock-prices,kaggle,4077,0,2034,False,False,False
9,leukipp/reddit-finance-data,kaggle,3909,0,1701,False,False,False


> **Limitations of this list**
>
> 1. **Truncated by `TOP_K` per platform.** Only the top-ranked `TOP_K` candidates *from each platform* (Kaggle, HuggingFace) are retained; lower-ranked matches are dropped (see the matched-vs-retained counts printed above). Raise `TOP_K` in the config to see more.
> 2. **Coarse title-keyword funnel.** When the search APIs do not expose column schemas, `has_engagement_metrics` / `has_sentiment_fields` / `is_complete` are inferred from titles/tags, not real columns, so these flags can have false positives and misses.
> 3. **Draft only.** Treat this as a shortlist to validate in Phase 2/3, not as ground truth.

---
## 6. Write artifact (CSV)

Writes the ranked draft candidate list to `src/eda/output/candidates.csv` (flattened).

In [20]:
# Flattened candidates CSV (the Phase 1 output).
candidates_csv = OUTPUT_DIR / "candidates.csv"
if ranked:
    cand_flat = pd.DataFrame([asdict(d) for d in ranked]).copy()
    cand_flat["columns"] = cand_flat["columns"].map(lambda c: "|".join(c) if isinstance(c, list) else c)
    cand_flat["date_range"] = cand_flat["date_range"].map(
        lambda d: f"{d[0]}..{d[1]}" if isinstance(d, (list, tuple)) else d
    )
    cand_flat.to_csv(candidates_csv, index=False)
else:
    pd.DataFrame(columns=["name", "source_platform"]).to_csv(candidates_csv, index=False)
print(f"Wrote CSV:  {candidates_csv}")

Wrote CSV:  d:\git\uol-bsc-cm3070-final-project\src\eda\output\candidates.csv


---
## Next step

Open **`02_highlevel_eval.ipynb`** to validate these draft candidates with cheap per-dataset stats and build a comparison matrix.